In [1]:
# Import necessary packages
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv



In [2]:
# Load the dataset
dataset = Planetoid(root='.', name='Cora')
data = dataset[0]

# Define the transformation to split links
# This is the key step for link prediction!
transform = T.RandomLinkSplit(
    num_val=0.1,  # 10% of edges for validation
    num_test=0.1, # 10% of edges for testing
    is_undirected=True,
    add_negative_train_samples=True, # Add negative edges for training
    neg_sampling_ratio=1.0 # 1 negative sample for each positive one
)

# Apply the transform
# This gives us three new Data objects
train_data, val_data, test_data = transform(data)


Processing...
Done!


In [3]:
# --- 2. Define the GNN Encoder ---
# We'll use a simple 2-layer GCN
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # We use the 'edge_index' from the *training* data for message passing
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# --- 3. Define the Link Prediction Decoder ---
# We'll use a simple dot product decoder
class LinkPredictor(torch.nn.Module):
    def forward(self, z, edge_label_index):
        # z: node embeddings [num_nodes, out_channels]
        # edge_label_index: edges to predict [2, num_edges]
        
        # Get the embeddings for the "source" and "target" nodes
        edge_feat_i = z[edge_label_index[0]]
        edge_feat_j = z[edge_label_index[1]]
        
        # Dot product
        return (edge_feat_i * edge_feat_j).sum(dim=-1)


In [4]:
# --- 4. Initialize Model and Optimizer ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCNEncoder(
    in_channels=dataset.num_node_features,
    hidden_channels=128,
    out_channels=64
).to(device)

decoder = LinkPredictor().to(device)

# Move all data to the GPU
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

# Initialize optimizer
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(decoder.parameters()), 
    lr=0.01
)
# We use Binary Cross Entropy with Logits
criterion = torch.nn.BCEWithLogitsLoss()


In [5]:

# --- 5. Training and Evaluation Functions ---
def train():
    model.train()
    decoder.train()
    optimizer.zero_grad()
    
    # Get node embeddings
    # We use all node features (data.x) and the training edges (train_data.edge_index)
    z = model(data.x, train_data.edge_index)
    
    # Get the predictions for the edges in the training set
    # train_data.edge_label_index includes *both* positive and negative edges
    out = decoder(z, train_data.edge_label_index)
    
    # train_data.edge_label holds the ground truth (1 for pos, 0 for neg)
    loss = criterion(out, train_data.edge_label)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(data_split):
    model.eval()
    decoder.eval()
    
    # Get node embeddings
    z = model(data.x, train_data.edge_index) # Always use train edges for encoding
    
    # Get predictions
    out = decoder(z, data_split.edge_label_index).sigmoid()
    
    # Compare to ground truth
    return roc_auc_score(data_split.edge_label.cpu().numpy(), out.cpu().numpy())


# --- 6. Run the Training Loop ---
for epoch in range(1, 101):
    loss = train()
    val_auc = test(val_data)
    test_auc = test(test_data)
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Test AUC: {test_auc:.4f}')

print("\nTraining complete!")
print(f'Final Test AUC: {test(test_data):.4f}')

Epoch: 010, Loss: 0.6774, Val AUC: 0.7324, Test AUC: 0.7419
Epoch: 020, Loss: 0.5845, Val AUC: 0.7931, Test AUC: 0.7662
Epoch: 030, Loss: 0.4906, Val AUC: 0.8201, Test AUC: 0.7934
Epoch: 040, Loss: 0.4344, Val AUC: 0.8410, Test AUC: 0.8171
Epoch: 050, Loss: 0.3776, Val AUC: 0.8463, Test AUC: 0.8246
Epoch: 060, Loss: 0.3240, Val AUC: 0.8388, Test AUC: 0.8303
Epoch: 070, Loss: 0.2720, Val AUC: 0.8301, Test AUC: 0.8301
Epoch: 080, Loss: 0.2196, Val AUC: 0.8155, Test AUC: 0.8221
Epoch: 090, Loss: 0.1685, Val AUC: 0.8052, Test AUC: 0.8140
Epoch: 100, Loss: 0.1198, Val AUC: 0.7865, Test AUC: 0.8066

Training complete!
Final Test AUC: 0.8066
